# Clasificador de textos en esperanto
**Segunda Guerra Mundial · Guerra Civil Española · Guerras Napoleónicas**

Minería de Textos · CUNEF · 2026/2027

## Qué hace este cuaderno

Dado un párrafo escrito en esperanto, decide de cuál de las tres guerras habla. Se construye en tres fases, de arriba abajo:

| Fase | Pregunta que responde | Qué se obtiene |
|---|---|---|
| **1 · Dataset** | ¿De dónde salen los ejemplos para aprender? | `data/dataset.csv`: 1.803 párrafos etiquetados (601 por clase) |
| **2 · NLP** | ¿Cómo convertimos un texto en palabras analizadas? | `EsperantoNLP`: tokenizador, etiquetador gramatical, lematizador y stop-words, hechos desde cero |
| **3 · Clasificación** | ¿Cómo aprende el modelo y cómo de bien lo hace? | Modelo entrenado, métricas y clasificación de páginas web reales |

## Cómo ejecutarlo

1. Ejecutar todo (*Entorno de ejecución → Ejecutar todas*). Tarda alrededor de un minuto, sin GPU y sin ficheros locales.
2. La variable `GENERAR_DATASET` de la primera celda elige entre cargar el CSV ya generado (`False`, valor por defecto) o reconstruirlo desde la API de Vikipedio (`True`, varios minutos por las pausas entre peticiones).

## Cómo leer el código

- Todas las librerías y parámetros están en la **primera celda de código**. Los nombres en MAYÚSCULAS son constantes que se pueden cambiar.
- Cada función lleva un *docstring* (el texto entre `"""` bajo su nombre) con **Parámetros**, **Devuelve**, **Cómo funciona** y, cuando ayuda, un **Ejemplo**. Con eso basta para usarla sin leer su interior.
- Los comentarios con `#` explican el porqué de cada decisión y qué guarda cada variable.

In [ ]:
# =============================================================================
# IMPORTACIONES Y PARÁMETROS GLOBALES
# =============================================================================
# Todo lo que se puede cambiar para modificar el comportamiento del cuaderno
# está en esta celda. El resto del cuaderno solo LEE estas variables.

# --- Librerías (todas vistas en clase) ---
import json                          # leer y guardar ficheros JSON (caché de los textos descargados)
import os                            # comprobar si existe un fichero y crear carpetas
import random                        # muestreo aleatorio reproducible al equilibrar las clases
import re                            # expresiones regulares: limpieza de texto y reglas del esperanto
import time                          # pausas entre peticiones para no saturar el servidor
from collections import Counter      # cuenta cuántas veces aparece cada elemento de una lista

import matplotlib.pyplot as plt      # gráficos
import numpy as np                   # vectores y matrices (palabras más discriminativas)
import pandas as pd                  # tablas (DataFrame)
import requests                      # peticiones HTTP (API de Vikipedio y páginas web)
from bs4 import BeautifulSoup        # extraer texto de una página HTML
from IPython.display import display  # mostrar tablas dentro de bucles y funciones

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

# --- Modo de ejecución ---
# False -> descarga el CSV ya generado desde GitHub (rápido). Valor por defecto.
# True  -> reconstruye el dataset desde la API de Vikipedio (varios minutos).
GENERAR_DATASET = False

# --- Reproducibilidad ---
# Semilla de todos los procesos aleatorios (equilibrado de clases, partición
# train/test, validación cruzada y modelos). Misma semilla = mismos resultados.
SEMILLA = 42

# --- Qué datos se piden y cómo se limpian ---
# Clases del problema: nombre de la clase -> categoría raíz en Vikipedio.
# Los nombres de categoría distinguen mayúsculas y minúsculas.
CATEGORIAS = {
    "Segunda Guerra Mundial": "Kategorio:Dua Mondmilito",
    "Guerra Civil Española": "Kategorio:Hispana Enlanda Milito",
    "Guerras Napoleónicas": "Kategorio:Napoleonaj militoj",
}
# Cuántos niveles de subcategorías se recorren: 0 = solo la categoría raíz,
# 1 = la raíz y sus subcategorías directas, 2 = también las subcategorías de estas...
PROFUNDIDAD = 1
# Subcategorías que NO se recorren (no hablan de la guerra en sí).
SUBCATS_EXCLUIDAS = {
    "Kategorio:Sekvoj de la Dua Mondmilito",   # consecuencias: es posguerra
    "Kategorio:Frankismo",                     # dictadura de 1939 a 1975: es posguerra
    "Kategorio:Brit-usona milito de 1812",     # es otra guerra
}
# Artículos concretos que se descartan.
ARTICULOS_EXCLUIDOS = {
    "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito",   # es una lista de nombres, no prosa
}
# Un párrafo es una línea de texto con al menos tantas palabras como indica esta constante.
MIN_PALABRAS = 10

# --- Conexión con la web ---
API_URL = "https://eo.wikipedia.org/w/api.php"   # API de MediaWiki de Vikipedio
# Wikimedia exige que cada programa se identifique con un User-Agent.
USER_AGENT = "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; práctica CUNEF)"
HEADERS = {"User-Agent": USER_AGENT}
PAUSA = 1.0              # segundos de cortesía entre peticiones (evita el error 429, "demasiadas peticiones")
MAX_REINTENTOS = 3       # veces que se intenta una petición antes de darla por fallida

# --- Ficheros ---
RUTA_CSV = "data/dataset.csv"          # dónde se guarda el dataset al generarlo
RUTA_TEXTOS = "data/raw/textos.json"   # caché de los textos descargados (evita repetir ~550 descargas)
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"

## Fase 1 · Dataset

**Qué se hace.** Con la API de Vikipedio (Wikipedia en esperanto) se piden los artículos de cada categoría, se descarga su texto plano, se parte en párrafos y se limpia. Cada párrafo hereda como etiqueta la clase de su artículo.

**Decisiones de diseño.** El estudio completo está en `docs/bitacora_exploracion_fase1.md` y `docs/estudio_profundidad_fase1.md`.

| Decisión | Motivo |
|---|---|
| **Tres guerras del mismo tema** | El enunciado pide clases cercanas. Las clases iniciales (WWI, WWII y Guerra Fría) compartían 32 artículos a profundidad 1, lo que mete ruido en las etiquetas. Otras candidatas (Vietnam, Afganistán, Aceh, guerra civil china, Corea) tenían pocos datos. |
| **Profundidad 1** (categoría + subcategorías directas) | Es la mínima con la que las Guerras Napoleónicas superan los 300 párrafos recomendados. La profundidad 2 les aporta solo 11 más y duplica la Segunda Guerra Mundial. |
| **Excluir 3 subcategorías y 1 artículo** | Son de posguerra, de otra guerra o una lista de nombres (ver `SUBCATS_EXCLUIDAS` y `ARTICULOS_EXCLUIDOS`). |
| **Descartar artículos en varias clases y párrafos repetidos** | Evita ejemplos con etiquetas contradictorias. Tras las exclusiones solo queda un artículo compartido (*Hugo Sperrle*). |
| **Equilibrar las clases** | Cada clase se recorta al tamaño de la menor, con muestreo aleatorio de semilla fija. Así el *accuracy* no premia a una clase mayoritaria. |
| **Conservar números y fechas** | Los años (1808, 1936, 1939...) casi delatan la clase. En la Fase 3 se entrena con y sin ellos para comparar. |

Párrafos únicos por clase según la profundidad (líneas de 10 palabras o más, sin artículos compartidos):

| Profundidad | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---:|---:|---:|
| 0 (solo la raíz) | 3.312 | 802 | 298 |
| **1 (elegida)** | 5.371 | 1.689 | 735 |
| 2 | 11.880 | 2.034 | 746 |

**Ruido de etiquetas.** Heredar la clase del artículo implica que algunos párrafos no hablan de la guerra (bibliografía, genealogía, enlaces de archivo). No se pueden separar a mano, así que se mide su efecto en los errores de la Fase 3.

### Acceso a la API de Vikipedio

In [ ]:
def pedir(url, params=None):
    """Hace una petición GET con reintentos y devuelve la respuesta, o None si falla.

    Parámetros
    ----------
    url : str
        Dirección a la que se pide (la API de Vikipedio o una página web).
    params : dict o None
        Parámetros de la URL (?clave=valor&...) escritos como diccionario.
        requests los convierte en texto por nosotros.

    Devuelve
    --------
    requests.Response si el servidor contesta con código 200; None en cualquier otro caso.

    Cómo funciona
    -------------
    1. Se intenta hasta MAX_REINTENTOS veces.
    2. Código 200: éxito, se devuelve la respuesta.
    3. Código 429 (demasiadas peticiones) o 5xx (fallo del servidor): son fallos pasajeros.
       Se espera el doble de tiempo en cada intento (1, 2, 4... segundos) y se reintenta.
    4. Otros códigos (403, 404...): reintentar no serviría, se devuelve None.
    5. Si no hay conexión (requests.RequestException) también se reintenta.

    Se devuelve None en lugar de lanzar un error para que cada llamada decida qué hacer.
    Así una web caída nunca rompe el cuaderno.
    """
    for intento in range(MAX_REINTENTOS):
        if intento > 0:                                    # antes de reintentar se espera
            time.sleep(PAUSA * 2 ** (intento - 1))
        try:
            respuesta = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if respuesta.status_code == 200:
                return respuesta
            if respuesta.status_code != 429 and respuesta.status_code < 500:
                return None                                # 403, 404...: no vale la pena reintentar
        except requests.RequestException:
            pass                                           # fallo de red: se reintenta
    return None


def consultar_api(params):
    """Hace una consulta a la API de MediaWiki (Vikipedio) y devuelve la respuesta como diccionario.

    Parámetros
    ----------
    params : dict
        Parámetros de la consulta según la documentación de MediaWiki, por ejemplo
        {"action": "query", "list": "categorymembers", "cmtitle": "Kategorio:Dua Mondmilito"}.

    Devuelve
    --------
    dict con el JSON de la respuesta (response.json() convierte el JSON en diccionarios y listas).

    Cómo funciona
    -------------
    Añade a los parámetros el formato JSON, llama a pedir(), comprueba que la API no haya
    devuelto un error y hace una pausa de cortesía. Lanza RuntimeError si no hay respuesta:
    aquí un fallo sí debe detener la generación, porque el dataset saldría incompleto.
    """
    params = {**params, "format": "json", "formatversion": 2}   # formatversion=2 devuelve listas, no diccionarios por ID
    respuesta = pedir(API_URL, params)
    if respuesta is None:
        raise RuntimeError(f"La API no respondió tras {MAX_REINTENTOS} intentos: {params}")
    datos = respuesta.json()
    if "error" in datos:                  # MediaWiki responde 200 incluso si la consulta es errónea
        raise RuntimeError(f"Error de la API de MediaWiki: {datos['error']['info']}")
    time.sleep(PAUSA)                     # pausa de cortesía con el servidor
    return datos

### Artículos de cada clase y descarga de su texto

In [ ]:
def miembros_de_categoria(categoria):
    """Pide a la API el contenido directo de una categoría.

    Parámetros
    ----------
    categoria : str
        Título de la categoría, por ejemplo "Kategorio:Dua Mondmilito".

    Devuelve
    --------
    (paginas, subcats) : dos listas de títulos.
        paginas = artículos de la categoría; subcats = sus subcategorías.

    Cómo funciona
    -------------
    La API devuelve como máximo 500 elementos por petición. Si hay más, la respuesta incluye
    la clave "continue" con los parámetros para pedir la página siguiente. El bucle sigue
    pidiendo hasta que esa clave desaparece. Cada elemento trae su espacio de nombres
    ("ns"): 0 es un artículo y 14 una subcategoría; se ignoran ficheros e imágenes.
    """
    params = {"action": "query", "list": "categorymembers", "cmtitle": categoria,
              "cmlimit": 500, "cmtype": "page|subcat"}
    paginas, subcats = [], []
    while True:
        datos = consultar_api(params)
        for miembro in datos["query"]["categorymembers"]:
            if miembro["ns"] == 0:
                paginas.append(miembro["title"])
            elif miembro["ns"] == 14:
                subcats.append(miembro["title"])
        if "continue" not in datos:        # no hay más páginas de resultados
            return paginas, subcats
        params.update(datos["continue"])   # pide la página siguiente


def articulos_de_categoria(categoria, profundidad=PROFUNDIDAD):
    """Devuelve el conjunto de artículos de una categoría y sus subcategorías.

    Parámetros
    ----------
    categoria : str
        Categoría de partida (la raíz de la clase).
    profundidad : int
        Niveles de subcategorías que se recorren. 0 = solo esta categoría.

    Devuelve
    --------
    set de títulos de artículos.

    Cómo funciona (recursión)
    -------------------------
    Se toman los artículos de la categoría. Si aún queda profundidad, se repite lo mismo
    en cada subcategoría (salvo las de SUBCATS_EXCLUIDAS) con un nivel menos de profundidad
    y se juntan los resultados. Un set no guarda repetidos.
    """
    paginas, subcats = miembros_de_categoria(categoria)
    articulos = set(paginas)
    if profundidad > 0:
        for sub in subcats:
            if sub not in SUBCATS_EXCLUIDAS:
                articulos |= articulos_de_categoria(sub, profundidad - 1)   # |= une conjuntos
    return articulos


def descargar_texto(titulo):
    """Descarga el texto plano de un artículo de Vikipedio.

    Parámetros
    ----------
    titulo : str
        Título exacto del artículo.

    Devuelve
    --------
    str con el texto completo (los títulos de sección aparecen como "== Título =="),
    o None si no se pudo descargar. La API devuelve un solo texto completo por petición.
    """
    params = {"action": "query", "prop": "extracts", "explaintext": 1,   # explaintext=1: texto plano, sin HTML
              "exsectionformat": "wiki", "redirects": 1, "titles": titulo,
              "format": "json", "formatversion": 2}
    respuesta = pedir(API_URL, params)
    if respuesta is None:
        return None
    try:
        return respuesta.json()["query"]["pages"][0].get("extract")
    except (ValueError, KeyError):         # respuesta que no es JSON o sin la estructura esperada
        return None


def descargar_textos(titulos):
    """Descarga el texto de todos los artículos y lo guarda en una caché local.

    Parámetros
    ----------
    titulos : list[str]
        Títulos de los artículos.

    Devuelve
    --------
    dict {título: texto o None}.

    Cómo funciona
    -------------
    Si ya existe RUTA_TEXTOS se lee de ahí y no se descarga nada (~550 peticiones).
    Si no, se descarga artículo a artículo con una pausa corta y se guarda el resultado
    en formato JSON para la próxima vez. El fichero es solo una comodidad: el cuaderno
    funciona igual sin él.
    """
    if os.path.exists(RUTA_TEXTOS):
        with open(RUTA_TEXTOS, "r", encoding="utf-8") as fichero:
            return json.load(fichero)

    textos = {}
    for i, titulo in enumerate(titulos, start=1):
        textos[titulo] = descargar_texto(titulo)
        time.sleep(0.2)
        if i % 50 == 0:
            print(f"  {i}/{len(titulos)} artículos descargados")

    os.makedirs(os.path.dirname(RUTA_TEXTOS), exist_ok=True)
    with open(RUTA_TEXTOS, "w", encoding="utf-8") as fichero:
        json.dump(textos, fichero, ensure_ascii=False)   # ensure_ascii=False conserva ĉ, ĝ, ŝ...
    return textos

### Partir en párrafos y limpiar

Un **párrafo** es una línea del texto plano con `MIN_PALABRAS` palabras o más, sin los títulos de sección (`== ... ==`). Se eliminan las etiquetas HTML, los enlaces y las referencias entre corchetes (`[1]`, pronunciaciones como `[kanAÚ]`) y se cambia `;` por `,` porque `;` es el separador del CSV. Se descartan las citas bibliográficas (ISBN/ISSN o *Autor (año).*), que no son prosa. La misma limpieza se aplica después a las páginas web que se quieren clasificar, para que el modelo vea texto igual al de entrenamiento.

In [ ]:
# Expresión regular que detecta líneas que NO son prosa:
#   \b(ISBN|ISSN)\b      las palabras ISBN o ISSN (citas de libros y revistas)
#   \(\d{4}\)[.,:]       un año entre paréntesis seguido de . , o :  -> "Autor (2011)."
#   \btraduko de teksto\b  aviso de traducción de Vikipedio
# "|" significa "o": basta con que se cumpla una de las tres.
REFERENCIA = re.compile(r"\b(ISBN|ISSN)\b|\(\d{4}\)[.,:]|\btraduko de teksto\b")


def parrafo_limpio(linea):
    r"""Limpia una línea de texto y devuelve el párrafo, o None si no vale como párrafo.

    Parámetros
    ----------
    linea : str
        Una línea de texto (de un artículo o de una página web).

    Devuelve
    --------
    str con el párrafo limpio, o None si es una cita bibliográfica o tiene menos de
    MIN_PALABRAS palabras (esto incluye las líneas vacías).

    Cómo funciona
    -------------
    re.sub(patrón, reemplazo, texto) sustituye todo lo que encaja con el patrón:
      <[^>]+>        un "<", uno o más caracteres que no sean ">", y un ">": etiqueta HTML
      https?://\S+   "http://" o "https://" y todo hasta el siguiente espacio: enlace
      \[[^\]]*\]     un "[", cualquier cosa que no sea "]", y un "]": referencia [1]
      \s+            uno o más espacios, tabuladores o saltos de línea: se dejan en un espacio
    """
    linea = re.sub(r"<[^>]+>", "", linea)         # etiquetas HTML
    linea = re.sub(r"https?://\S+", "", linea)    # enlaces sueltos
    linea = re.sub(r"\[[^\]]*\]", "", linea)      # referencias y pronunciaciones entre corchetes
    linea = linea.replace(";", ",")               # ";" es el separador del CSV
    linea = re.sub(r"\s+", " ", linea).strip()    # espacios repetidos -> uno; quita los de los extremos
    if REFERENCIA.search(linea) or len(linea.split()) < MIN_PALABRAS:
        return None
    return linea


def partir_en_parrafos(texto):
    """Parte el texto de un artículo en párrafos limpios.

    Parámetros
    ----------
    texto : str
        Texto plano de un artículo, con saltos de línea y títulos "== Sección ==".

    Devuelve
    --------
    list[str] con los párrafos que superan la limpieza, en el orden del artículo.
    """
    parrafos = []
    for linea in texto.split("\n"):                # cada salto de línea separa un párrafo
        if linea.strip().startswith("=="):         # título de sección: no es un párrafo
            continue
        parrafo = parrafo_limpio(linea)
        if parrafo is not None:
            parrafos.append(parrafo)
    return parrafos


# Prueba rápida (sin internet): el título se descarta, "[1]" se borra, ";" pasa a "," y la línea corta se descarta.
texto_ejemplo = ("== Historio ==\n"
                 "Napoleono naskiĝis en 1769 [1] kaj mortis en 1821; li estis granda militestro de Francio.\n"
                 "Tro mallonga.\n")
assert partir_en_parrafos(texto_ejemplo) == [
    "Napoleono naskiĝis en 1769 kaj mortis en 1821, li estis granda militestro de Francio."]

### Construir, validar y generar el dataset

Tras limpiar, se descartan los artículos que están en más de una clase y los párrafos repetidos (dentro de una clase o entre clases). Después se recorta cada clase al tamaño de la menor y se mezcla el conjunto. El resultado cumple el contrato de datos del enunciado: CSV en UTF-8, separador `;`, columnas `text;class`, un párrafo por fila y sin `;` dentro del texto.

In [ ]:
def construir_dataset(titulos, textos, articulos_por_clase):
    """Convierte los textos descargados en un dataset equilibrado de párrafos etiquetados.

    Parámetros
    ----------
    titulos : list[str]
        Títulos de los artículos, ordenados alfabéticamente.
    textos : dict
        {título: texto del artículo o None}, el resultado de descargar_textos().
    articulos_por_clase : dict
        {clase: set de títulos}, el resultado de articulos_de_categoria() para cada clase.

    Devuelve
    --------
    pandas.DataFrame con las columnas "text" y "class", mezclado y con las mismas filas por clase.

    Cómo funciona
    -------------
    1. Para cada artículo se miran sus clases. Si pertenece a más de una, se descarta.
       Si pertenece a una, sus párrafos se añaden a la lista de esa clase.
    2. Se quitan los párrafos repetidos dentro de cada clase. dict.fromkeys(lista) elimina
       repetidos conservando el orden.
    3. Se cuenta cuántas veces aparece cada párrafo en el conjunto de todas las clases y se
       quitan los que aparecen más de una vez: serían ejemplos con etiquetas contradictorias.
    4. Equilibrar: se calcula n_min, el tamaño de la clase menor, y de cada clase se toman
       n_min párrafos al azar (random.Random(SEMILLA) hace el azar reproducible).
    5. Se juntan las tres clases y se mezclan las filas (df.sample(frac=1) = todas, en otro orden).
    """
    # 1) Reparto de los párrafos por clase
    por_clase = {clase: [] for clase in CATEGORIAS}
    for titulo in titulos:
        if textos.get(titulo) is None:                      # no se pudo descargar
            continue
        clases = [c for c in CATEGORIAS if titulo in articulos_por_clase[c]]
        if len(clases) > 1:
            print("Artículo compartido, se descarta:", titulo)
            continue
        por_clase[clases[0]].extend(partir_en_parrafos(textos[titulo]))

    # 2) Sin repetidos dentro de cada clase
    por_clase = {clase: list(dict.fromkeys(ps)) for clase, ps in por_clase.items()}

    # 3) Sin párrafos idénticos entre clases distintas
    veces = Counter(p for ps in por_clase.values() for p in ps)   # {párrafo: nº de veces}
    print("Párrafos idénticos en varias clases, descartados:", sum(1 for n in veces.values() if n > 1))
    por_clase = {clase: [p for p in ps if veces[p] == 1] for clase, ps in por_clase.items()}
    print("Antes de equilibrar:", {clase: len(ps) for clase, ps in por_clase.items()})

    # 4) Equilibrar al tamaño de la clase menor
    n_min = min(len(ps) for ps in por_clase.values())
    azar = random.Random(SEMILLA)
    equilibrado = {clase: azar.sample(ps, n_min) for clase, ps in por_clase.items()}

    # 5) Una fila por párrafo y filas mezcladas
    filas = [(parrafo, clase) for clase, ps in equilibrado.items() for parrafo in ps]
    df = pd.DataFrame(filas, columns=["text", "class"])
    return df.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)


def validar_dataset(df):
    """Comprueba que el dataset cumple el contrato de datos. Lanza AssertionError si no.

    Parámetros
    ----------
    df : pandas.DataFrame
        Dataset con las columnas "text" y "class".

    Cada assert exige una condición; si es falsa, el cuaderno se detiene con el mensaje indicado.
    """
    assert list(df.columns) == ["text", "class"], "las columnas deben ser text;class"
    assert df["text"].notna().all() and (df["text"].str.strip() != "").all(), "hay párrafos vacíos"
    assert set(df["class"]) == set(CATEGORIAS), "las clases del CSV no coinciden con CATEGORIAS"
    assert df["class"].value_counts().nunique() == 1, "las clases no están equilibradas"
    assert not df["text"].str.contains(";").any(), "hay un ';' dentro de un párrafo"
    assert not df["text"].duplicated().any(), "hay párrafos repetidos"
    assert (df["text"].str.split().str.len() >= MIN_PALABRAS).all(), "hay párrafos demasiado cortos"


def generar_dataset():
    """Genera el dataset completo desde la API de Vikipedio y lo guarda en RUTA_CSV.

    Devuelve
    --------
    pandas.DataFrame con el dataset releído desde el CSV (así se comprueba también que el
    fichero guardado cumple el contrato).

    Cómo funciona
    -------------
    Encadena los pasos de arriba: artículos de cada clase -> descarga de textos ->
    construir_dataset -> guardar CSV -> releer y validar.
    """
    articulos_por_clase = {clase: articulos_de_categoria(raiz) for clase, raiz in CATEGORIAS.items()}
    # Unión de las tres clases, sin los artículos excluidos y ordenada alfabéticamente.
    titulos = sorted(set().union(*articulos_por_clase.values()) - ARTICULOS_EXCLUIDOS)
    print(f"{len(titulos)} artículos distintos")

    textos = descargar_textos(titulos)
    df = construir_dataset(titulos, textos, articulos_por_clase)

    os.makedirs(os.path.dirname(RUTA_CSV), exist_ok=True)
    df.to_csv(RUTA_CSV, sep=";", index=False, encoding="utf-8")
    return pd.read_csv(RUTA_CSV, sep=";", encoding="utf-8")

### Cargar o generar el dataset

Según `GENERAR_DATASET`, el dataset se construye con las funciones anteriores (`True`) o se descarga del repositorio (`False`). En los dos casos queda en la variable `df`, con las columnas `text` y `class`.

In [ ]:
if GENERAR_DATASET:
    df = generar_dataset()
else:
    df = pd.read_csv(URL_CSV_REMOTO, sep=";", encoding="utf-8")   # sep=";" porque es el separador del CSV

validar_dataset(df)
print(df.shape)
print(df["class"].value_counts())
df.head()

## Fase 2 · NLP

### Modelo de NLP para esperanto (basado en reglas)

**Objetivo.** Construir desde cero, sin librerías de NLP, un modelo con una API parecida a la de spaCy: `nlp = EsperantoNLP()` crea el modelo, `doc = nlp(texto)` lo aplica y devuelve un `Doc` formado por objetos `Token` con `text`, `lemma_`, `pos_`, `morph` e `is_stop`.

**Por qué funciona con reglas.** El esperanto es regular: la terminación de una palabra fija su categoría gramatical (`-o` sustantivo, `-a` adjetivo, `-e` adverbio, `-as/-is/-os/-us/-u/-i` verbo), y el plural (`-j`) y el acusativo (`-n`) se añaden siempre igual. El análisis de cada palabra sigue este orden y gana la primera regla que encaja:

1. **Pronombre con acusativo** (`min`, `vin`...).
2. **Nombre propio**: palabra con mayúscula que no abre frase, o escrita toda en mayúsculas.
3. **Palabras cerradas** (artículo, pronombres, preposiciones, conjunciones, partículas, adverbios e interjecciones) y **numerales** (`du`, `dudek`). Son listas finitas.
4. **Ordinales** (`dua`), **posesivos** (`mia`) y **correlativos** (`kiu`, `tie`, `iam`...), que siguen un patrón.
5. **Participios** (`-ant-`, `-int-`, `-ont-`, `-at-`, `-it-`, `-ot-`): se lematizan con el infinitivo del verbo.
6. **Verbos**, por la terminación.
7. **Sustantivos, adjetivos y adverbios**, por `-o`, `-a`, `-e`.
8. Si nada encaja, categoría `X` (palabra extranjera, abreviatura...).

**Etiquetas** (Universal Dependencies, las mismas que usa spaCy): `NOUN` sustantivo · `PROPN` nombre propio · `VERB` verbo · `ADJ` adjetivo · `ADV` adverbio · `DET` determinante · `PRON` pronombre · `ADP` preposición · `CCONJ`/`SCONJ` conjunción coordinante/subordinante · `PART` partícula · `INTJ` interjección · `NUM` número · `PUNCT` puntuación · `X` desconocido.

**Lema.** Es la forma de diccionario: se quitan `-j` y `-n` y el verbo queda en infinitivo (`-i`), el sustantivo en `-o`... siempre en minúsculas. Así `soldato` y `soldatoj` cuentan como la misma palabra en la Fase 3.

**Stop-words.** Palabras muy frecuentes que no distinguen un tema (`la`, `de`, `kaj`...). Se cargan de `vortoj.json` (repositorio de la asignatura), quedándose con las marcadas como `ignorinda`. Si la descarga falla se usa una lista de respaldo, así el cuaderno no depende de esa web.

**Limitaciones conocidas.**
- Sin diccionario, una raíz que termina como un participio (`milito`, `soldato`, `kandidato`) se confundiría con uno. Se resuelve con una lista corta de excepciones; `milito` es la palabra clave de este corpus, así que no podía fallar.
- Un nombre propio que abre frase se etiqueta por su terminación (suele salir como sustantivo).
- Las palabras extranjeras que no siguen la morfología del esperanto quedan como `X`.

In [ ]:
# =============================================================================
# FASE 2 · DATOS DE LA GRAMÁTICA DEL ESPERANTO
# =============================================================================
# Esta celda solo DEFINE datos (listas de palabras y patrones). La lógica que los
# usa está en la función analizar(), en la celda siguiente.

# Dónde está la lista de stop-words de la asignatura.
URL_STOPWORDS = "https://raw.githubusercontent.com/jparisu/nlp-esperantilo/main/resources/esperanto/vortoj.json"

# --- 1. Palabras cerradas ---
# Son listas finitas. Clave = etiqueta gramatical; valor = palabras separadas por espacios
# (más abajo se pasan a lista con .split()). El orden de las claves importa solo si una
# palabra apareciera en dos listas: gana la última.
PALABRAS_CERRADAS = {
    "DET":   "la ambaŭ",
    "PRON":  "mi vi li ŝi ĝi ni ili oni si ci",
    "ADP":   ("al anstataŭ antaŭ apud da de dum ekde ekster el en inter je kontraŭ krom kun laŭ "
              "malantaŭ malgraŭ per po por post preter pri pro sen sub super sur tra trans ĉe ĉirkaŭ ĝis"),
    "CCONJ": "kaj aŭ sed nek do tamen",
    "SCONJ": "ke se ĉar ol kvankam kvazaŭ",
    "PART":  "ne ĉu jes ja jen ĉi ajn ju des",
    "ADV":   ("ankaŭ ankoraŭ apenaŭ baldaŭ eĉ for hieraŭ hodiaŭ jam morgaŭ nun nur plej pli plu "
              "preskaŭ tre tro tuj tute ĵus nepre mem"),
    "INTJ":  "ha he ho hu nu ve adiaŭ",
}

# Diccionario inverso {palabra: etiqueta}: es lo que consulta analizar().
# Por ejemplo CERRADAS["kaj"] vale "CCONJ".
CERRADAS = {}
for etiqueta, palabras in PALABRAS_CERRADAS.items():
    for palabra in palabras.split():
        CERRADAS[palabra] = etiqueta

PRONOMBRES = set(PALABRAS_CERRADAS["PRON"].split())     # mi, vi, li, ŝi, ĝi, ni, ili, oni, si, ci

# --- 2. Stop-words ---
# Posesivos (mia, via, lia...): pronombre + "a". vortoj.json no los incluye aunque son palabras funcionales.
POSESIVOS = {pronombre + "a" for pronombre in PRONOMBRES}
STOPWORDS_EXTRA = POSESIVOS | {"ambaŭ"}                 # se añaden siempre a las de vortoj.json

# --- 3. Numerales, ordinales y correlativos: siguen patrones regulares ---
# Los numerales se pegan entre sí: du (2), dudek (20), tricent (300), mil.
NUMEROS = "nul unu du tri kvar kvin ses sep ok naŭ dek cent mil".split()
_NUM = "(?:" + "|".join(NUMEROS) + ")"        # "(?:nul|unu|du|...)" = "una cualquiera de estas palabras"

# Lista de respaldo si falla la descarga de vortoj.json: palabras cerradas, posesivos y numerales básicos.
STOPWORDS_RESPALDO = set(CERRADAS) | STOPWORDS_EXTRA | set(NUMEROS)

# En las expresiones regulares: ^ = inicio de la palabra, $ = final, + = "una o más veces".
NUMERAL = re.compile(rf"^{_NUM}+$")           # du, dudek, tricent, mil
ORDINAL = re.compile(rf"^{_NUM}+a$")          # dua, deka, mila (el numeral + "a")

# Correlativos: prefijo (ki-, ti-, i-, ĉi-, neni-) + terminación (-u, -o, -a, -es, -e, -am, -al, -el, -om).
# Ejemplos: kiu (cuál), tio (eso), ĉiam (siempre), nenies (de nadie).
CORRELATIVO = re.compile(r"^(?:ki|ti|i|ĉi|neni)(u|o|a|es|e|am|al|el|om)$")
# Categoría de cada correlativo según su terminación.
POS_CORRELATIVO = {"u": "PRON", "o": "PRON", "a": "DET", "es": "DET",
                   "e": "ADV", "am": "ADV", "al": "ADV", "el": "ADV", "om": "ADV"}

# --- 4. Participios ---
# raíz del verbo (mínimo 3 letras) + sufijo de participio + vocal final.
#   sufijos activos: -ant- (presente), -int- (pasado), -ont- (futuro)
#   sufijos pasivos: -at-  (presente), -it-  (pasado), -ot-  (futuro)
#   vocal final: -a adjetivo, -o sustantivo, -e adverbio
# (?P<rad>...) crea un grupo con nombre para recuperarlo después como m["rad"].
# ".{3,}?" = al menos 3 caracteres cualesquiera, los menos posibles.
PARTICIPIO = re.compile(r"^(?P<rad>.{3,}?)(?P<part>ant|int|ont|at|it|ot)(?P<fin>[aoe])$")
# Sin diccionario, estas raíces parecerían un participio sin serlo (milito != mili + -it- + -o).
FALSOS_PARTICIPIOS = (
    "milit", "soldat", "kandidat", "diplomat", "demokrat", "aristokrat", "advokat", "magistrat", "senat",
    "pirat", "mandat", "komitat", "primat", "traktat", "vizit", "kredit", "profit", "limit", "ermit",
    "granit", "dinamit", "plant", "elefant", "diamant", "gigant", "horizont", "front", "kvant",
)

# --- 5. Verbos ---
# raíz (mínimo 2 letras) + terminación. La raíz + "i" es el infinitivo, que es el lema.
VERBO = re.compile(r"^(?P<rad>.{2,})(?P<fin>as|is|os|us|u|i)$")
TIEMPO_VERBO = {"as": "Pres", "is": "Past", "os": "Fut", "us": "Cond", "u": "Imp"}

In [ ]:
def cargar_stopwords(url=URL_STOPWORDS):
    """Descarga las stop-words de la asignatura (vortoj.json).

    Parámetros
    ----------
    url : str
        Dirección del fichero vortoj.json.

    Devuelve
    --------
    set de palabras en minúsculas. Si la descarga falla, STOPWORDS_RESPALDO.

    Cómo funciona
    -------------
    vortoj.json es un diccionario con una lista "vortoj" de entradas {"vorto": palabra,
    "ignorinda": True/False, ...}. Se guardan las palabras marcadas como "ignorinda"
    (en esperanto, "que se debe ignorar").
    """
    respuesta = pedir(url)
    if respuesta is not None:
        try:
            palabras = {p["vorto"].lower() for p in respuesta.json()["vortoj"] if p.get("ignorinda")}
            if palabras:
                return palabras
        except (ValueError, KeyError, TypeError):
            pass
    print("Aviso: no se pudo descargar vortoj.json; se usa la lista de respaldo.")
    return set(STOPWORDS_RESPALDO)


def separar_flexion(palabra):
    """Quita el acusativo (-n) y el plural (-j) de una palabra en minúsculas.

    Parámetros
    ----------
    palabra : str
        Palabra en minúsculas, por ejemplo "urbojn" (las ciudades, como complemento directo).

    Devuelve
    --------
    (base, morf) : la palabra sin esas terminaciones y un diccionario con los rasgos hallados.
        "urbojn" -> ("urbo", {"Number": "Plur", "Case": "Acc"})
        "urbo"   -> ("urbo", {})

    Cómo funciona
    -------------
    El acusativo es una -n tras una vocal o -j (-on, -an, -en, -un, -jn) y el plural es una -j
    tras -o, -a o -u. Se exige len > 3 para no recortar palabras cortas ("ŝin", "tuj").
    Se quita primero la -n y después la -j, porque en esperanto van en ese orden (-oj-n).
    """
    base = palabra
    acusativo = len(base) > 3 and base.endswith("n") and base[-2] in "oaeuj"
    if acusativo:
        base = base[:-1]
    plural = len(base) > 3 and base.endswith("j") and base[-2] in "oau"
    if plural:
        base = base[:-1]

    morf = {}
    if plural:
        morf["Number"] = "Plur"
    if acusativo:
        morf["Case"] = "Acc"
    return base, morf


def analizar(palabra, inicio_frase=False):
    """Análisis morfológico de UNA palabra: categoría, lema y rasgos.

    Parámetros
    ----------
    palabra : str
        La palabra tal como aparece en el texto (con sus mayúsculas).
    inicio_frase : bool
        True si es la primera palabra de una frase. Una mayúscula inicial indica nombre propio
        solo cuando NO está al principio de la frase.

    Devuelve
    --------
    (pos, lema, morf) : etiqueta gramatical (str), lema en minúsculas (str) y rasgos (dict).
        analizar("soldatoj") -> ("NOUN", "soldato", {"Number": "Plur"})
        analizar("batalis")  -> ("VERB", "batali", {"VerbForm": "Fin", "Tense": "Past"})

    Cómo funciona
    -------------
    Se prueban las reglas en el orden de la celda de texto de la Fase 2 y se devuelve el
    resultado de la primera que encaja (por eso el orden importa). Ejemplo: "milito" se
    comprueba como participio antes que como sustantivo, y por eso necesita la lista
    FALSOS_PARTICIPIOS.
    """
    w = palabra.lower()

    # Regla 1: pronombre con acusativo (min, vin, lin, ŝin...): se quita la -n.
    if w.endswith("n") and w[:-1] in PRONOMBRES:
        return "PRON", w[:-1], {"Case": "Acc"}

    base, morf = separar_flexion(w)     # base = sin -n/-j; morf = {"Number": ..., "Case": ...}

    # Regla 2: nombre propio = mayúscula que no abre frase, o palabra toda en mayúsculas (CNT, SS).
    if (palabra[0].isupper() and not inicio_frase) or (len(palabra) > 1 and palabra.isupper()):
        return "PROPN", base, morf

    # Regla 3: palabras cerradas y numerales. Se miran sin quitar -n/-j (por eso usan w, no base).
    if w in CERRADAS:
        return CERRADAS[w], w, {}
    if NUMERAL.match(w):
        return "NUM", w, {}

    # Regla 4: ordinales, posesivos y correlativos.
    if ORDINAL.match(base):                                  # dua, deka...
        return "ADJ", base, {**morf, "NumType": "Ord"}       # {**a, "k": v} = copia de a con una clave más
    if base.endswith("a") and base[:-1] in PRONOMBRES:       # mia, via, lia...
        return "DET", base, {**morf, "Poss": "Yes"}
    m = CORRELATIVO.match(base)                              # kiu, tio, ĉiam...
    if m:
        return POS_CORRELATIVO[m.group(1)], base, morf       # group(1) = la terminación (u, o, am...)

    # Regla 5: participios. La vocal final da la categoría; el lema es el infinitivo del verbo.
    m = PARTICIPIO.match(base)
    if m and not (m["rad"] + m["part"]).endswith(FALSOS_PARTICIPIOS):
        parte = m["part"]                                    # ant, int, ont, at, it o ot
        morf.update({"VerbForm": "Part",
                     "Voice": "Act" if parte.endswith("nt") else "Pass",
                     "Tense": {"a": "Pres", "i": "Past", "o": "Fut"}[parte[0]]})
        return {"a": "ADJ", "o": "NOUN", "e": "ADV"}[m["fin"]], m["rad"] + "i", morf

    # Regla 6: verbos. Lema = raíz + "i".
    m = VERBO.match(base)
    if m:
        if m["fin"] == "i":
            morf["VerbForm"] = "Inf"
        else:
            morf.update({"VerbForm": "Fin", "Tense": TIEMPO_VERBO[m["fin"]]})
        return "VERB", m["rad"] + "i", morf

    # Regla 7: sustantivos, adjetivos y adverbios por su vocal final.
    if len(base) >= 3 and base[-1] in "oae":
        pos = {"o": "NOUN", "a": "ADJ", "e": "ADV"}[base[-1]]
        if pos != "ADV":
            morf.setdefault("Number", "Sing")                # si no era plural, es singular
        return pos, base, morf

    # Regla 8: nada encaja (palabra extranjera, abreviatura...).
    return "X", base, morf

In [ ]:
# =============================================================================
# FASE 2 · API TIPO spaCy: Token, Doc y EsperantoNLP
# =============================================================================

# Tras estos signos empieza una frase nueva: la mayúscula de la palabra siguiente
# NO indica nombre propio.
FIN_FRASE = {".", "!", "?", "…", ":", "«", "“", "„", '"'}

# Expresión regular del tokenizador. Un token es UNA de estas tres cosas (el "|" separa las
# alternativas y se prueban en orden). Con re.VERBOSE se permiten espacios y comentarios.
#   [^\W\d_]  = una letra (cualquier carácter que no sea no-alfanumérico, ni dígito, ni "_")
PATRON_TOKEN = re.compile(r"""
      \d+(?:[.,]\d+)*(?:[-‑]?a[jn]{0,2}(?![^\W\d_]))?   # número (1939, 3,5), con ordinal opcional (17-a, 1930-aj, 8an, 20a)
    | [^\W\d_]+(?:['’-][^\W\d_]+)*                       # palabra, con guion o apóstrofo interno (Brit-usona)
    | [^\w\s]                                                 # cualquier otro signo: puntuación, comillas...
""", re.VERBOSE)


class Token:
    """Una palabra, número o signo del texto con su análisis (equivale a spacy.tokens.Token).

    Atributos
    ---------
    text : str      forma original, tal como aparece en el texto
    i : int         posición del token en el Doc (0, 1, 2...)
    idx : int       posición de su primer carácter en el texto original
    pos_ : str      categoría gramatical ("NOUN", "VERB", "PUNCT"...); "X" si no se conoce
    lemma_ : str    forma de diccionario en minúsculas ("soldatoj" -> "soldato")
    morph : dict    rasgos morfológicos ({"Number": "Plur", "Case": "Acc"}, {"Tense": "Past"}...)
    is_stop : bool  True si es una stop-word
    """

    def __init__(self, text, i, idx):
        self.text = text
        self.i = i
        self.idx = idx
        self.pos_ = "X"          # se rellenan al analizar el token
        self.lemma_ = ""
        self.morph = {}
        self.is_stop = False

    @property
    def is_punct(self):
        """True si el token es un signo de puntuación (como token.is_punct de spaCy)."""
        return self.pos_ == "PUNCT"

    def __str__(self):
        return self.text


class Doc:
    """Resultado de analizar un texto: una secuencia de Token (equivale a spacy.tokens.Doc).

    Se comporta como una lista: len(doc) da el número de tokens, doc[0] el primero y
    "for token in doc" los recorre.
    """

    def __init__(self, text, tokens):
        self.text = text         # texto original completo
        self.tokens = tokens     # lista de objetos Token

    def __len__(self):
        return len(self.tokens)

    def __iter__(self):
        return iter(self.tokens)

    def __getitem__(self, i):
        return self.tokens[i]

    def __repr__(self):
        return f"Doc({len(self)} tokens): {self.text[:60]!r}"

    def to_dataframe(self):
        """Tabla con un token por fila, para inspeccionar el análisis."""
        return pd.DataFrame([{"texto": t.text, "pos": t.pos_, "lema": t.lemma_,
                              "morfología": t.morph, "stop": t.is_stop} for t in self.tokens])


class EsperantoNLP:
    """Pipeline de NLP para esperanto basado en reglas (equivale a spacy.Language).

    Uso, igual que en spaCy:
        nlp = EsperantoNLP()                          # crea el modelo
        doc = nlp("La soldatoj batalis.")             # analiza un texto -> Doc
        [t.lemma_ for t in doc]                       # ['la', 'soldato', 'batali', '.']

    Atributo
    --------
    stopwords : set    palabras que se marcan como stop-words (se pueden modificar)
    """

    def __init__(self, stopwords=None):
        # Sin argumento: stop-words de vortoj.json más los posesivos. También se puede pasar un conjunto propio.
        self.stopwords = set(stopwords) if stopwords is not None else cargar_stopwords() | STOPWORDS_EXTRA

    # --- Gestión de stop-words ---
    def add_stopword(self, palabra):
        """Añade una stop-word."""
        self.stopwords.add(palabra.lower())

    def remove_stopword(self, palabra):
        """Quita una stop-word (no falla si no estaba)."""
        self.stopwords.discard(palabra.lower())

    # --- Tokenizador ---
    def tokenizar(self, texto):
        """Parte el texto en tokens con PATRON_TOKEN.

        Devuelve una lista de (forma, posición): tokenizar("Hola, mundo") -> [("Hola", 0), (",", 4), ("mundo", 6)].
        finditer recorre todas las coincidencias; m.group() es el texto y m.start() su posición.
        """
        return [(m.group(), m.start()) for m in PATRON_TOKEN.finditer(texto)]

    # --- Pipeline completo ---
    def __call__(self, texto):
        """Analiza un texto: tokeniza, etiqueta, lematiza y marca las stop-words.

        Parámetros
        ----------
        texto : str
            Texto en esperanto.

        Devuelve
        --------
        Doc con un Token por cada palabra, número o signo.

        Cómo funciona
        -------------
        Se recorren los tokens en orden con la variable inicio_frase (True al empezar y tras
        cada signo de FIN_FRASE). Según el primer carácter de cada token:
          - dígito: número. Su lema es la parte numérica ("17-an" -> "17").
          - letra:  palabra. Se llama a analizar(); es stop-word si lo es la forma escrita o
                    su lema (así "estis" cuenta como stop-word porque "esti" lo es).
          - otro:   puntuación.
        """
        tokens = []
        inicio_frase = True
        for forma, idx in self.tokenizar(texto):
            token = Token(text=forma, i=len(tokens), idx=idx)
            if forma[0].isdigit():
                token.pos_ = "NUM"
                token.lemma_ = re.match(r"\d+(?:[.,]\d+)*", forma).group()
                inicio_frase = False
            elif forma[0].isalpha():
                token.pos_, token.lemma_, token.morph = analizar(forma, inicio_frase)
                token.is_stop = forma.lower() in self.stopwords or token.lemma_ in self.stopwords
                inicio_frase = False
            else:
                token.pos_, token.lemma_ = "PUNCT", forma
                if forma in FIN_FRASE:
                    inicio_frase = True
            tokens.append(token)
        return Doc(texto, tokens)

    def pipe(self, textos):
        """Analiza varios textos, uno tras otro (como nlp.pipe de spaCy). Devuelve un generador de Doc."""
        for texto in textos:
            yield self(texto)

### Pruebas del modelo

Cada caso de `PRUEBAS` fija el análisis esperado de una frase. `assert condición` detiene el cuaderno con un error si la condición es falsa, así que si el modelo cambia y rompe algo se nota al instante. Los casos cubren nombres propios, participios, la raíz `milit-`, ordinales, correlativos y acusativos.

In [ ]:
nlp = EsperantoNLP()
print(f"Stop-words cargadas: {len(nlp.stopwords)}")


def analisis(texto):
    """Lista de (forma, categoría, lema) de un texto, sin los signos de puntuación."""
    return [(t.text, t.pos_, t.lemma_) for t in nlp(texto) if not t.is_punct]


# frase -> análisis esperado [(forma, categoría, lema), ...]
PRUEBAS = {
    "La soldatoj de Napoleono batalis.": [
        ("La", "DET", "la"), ("soldatoj", "NOUN", "soldato"), ("de", "ADP", "de"),
        ("Napoleono", "PROPN", "napoleono"), ("batalis", "VERB", "batali")],
    "Milito estas terura.": [
        ("Milito", "NOUN", "milito"), ("estas", "VERB", "esti"), ("terura", "ADJ", "terura")],
    "Batalantaj trupoj okupis urbojn.": [
        ("Batalantaj", "ADJ", "batali"), ("trupoj", "NOUN", "trupo"),
        ("okupis", "VERB", "okupi"), ("urbojn", "NOUN", "urbo")],
    "En 1939 la dua mondmilito komenciĝis.": [
        ("En", "ADP", "en"), ("1939", "NUM", "1939"), ("la", "DET", "la"), ("dua", "ADJ", "dua"),
        ("mondmilito", "NOUN", "mondmilito"), ("komenciĝis", "VERB", "komenciĝi")],
    "Ili vidis ĝin tie kaj poste.": [
        ("Ili", "PRON", "ili"), ("vidis", "VERB", "vidi"), ("ĝin", "PRON", "ĝi"),
        ("tie", "ADV", "tie"), ("kaj", "CCONJ", "kaj"), ("poste", "ADV", "poste")],
    "La 17-an de aŭgusto Hispanio perdis du batalojn.": [
        ("La", "DET", "la"), ("17-an", "NUM", "17"), ("de", "ADP", "de"), ("aŭgusto", "NOUN", "aŭgusto"),
        ("Hispanio", "PROPN", "hispanio"), ("perdis", "VERB", "perdi"), ("du", "NUM", "du"),
        ("batalojn", "NOUN", "batalo")],
    "Dum la milito soldatoj estis mortigitaj.": [
        ("Dum", "ADP", "dum"), ("la", "DET", "la"), ("milito", "NOUN", "milito"),
        ("soldatoj", "NOUN", "soldato"), ("estis", "VERB", "esti"), ("mortigitaj", "ADJ", "mortigi")],
}
for texto, esperado in PRUEBAS.items():
    obtenido = analisis(texto)
    assert obtenido == esperado, f"Fallo en: {texto}\n  esperado: {esperado}\n  obtenido: {obtenido}"

# Stop-words: se pueden consultar, añadir y quitar.
assert nlp("la soldato")[0].is_stop and not nlp("la soldato")[1].is_stop
nlp.add_stopword("soldato")
assert nlp("soldato")[0].is_stop
nlp.remove_stopword("soldato")
assert not nlp("soldato")[0].is_stop

# Stop-words por lema (estis -> esti), posesivos y ordinales escritos sin guion ("la 8an de majo").
assert nlp("estis")[0].is_stop and nlp("mia")[0].is_stop and nlp("ambaŭ")[0].is_stop
assert analisis("la 8an de majo") == [("la", "DET", "la"), ("8an", "NUM", "8"), ("de", "ADP", "de"), ("majo", "NOUN", "majo")]

# Rasgos morfológicos.
assert nlp("urbojn")[0].morph == {"Number": "Plur", "Case": "Acc"}
assert nlp("batalis")[0].morph == {"VerbForm": "Fin", "Tense": "Past"}
print(f"✔ {len(PRUEBAS)} frases de prueba y comprobaciones de stop-words y morfología superadas")

# Ejemplo visual: una tabla con un token por fila.
doc = nlp("La soldatoj de Napoleono batalis kontraŭ la hispanaj armeoj en 1808.")
print(doc)
display(doc.to_dataframe())

### El modelo aplicado a todo el corpus

Se analizan los 1.803 párrafos. La tabla `tokens` guarda un token por fila para poder contar categorías y lemas. Dos comprobaciones de sentido: qué proporción de tokens queda sin categoría (`X`) y cuáles son los lemas de contenido más frecuentes de cada clase.

In [ ]:
t0 = time.time()
docs = [nlp(texto) for texto in df["text"]]      # lista de Doc: uno por párrafo, en el mismo orden que df
print(f"{len(docs):,} párrafos · {sum(len(d) for d in docs):,} tokens · {time.time() - t0:.1f} s")

# Una fila por token: su categoría, si es stop-word, su lema y la clase del párrafo del que sale.
tokens = pd.DataFrame(
    [(t.pos_, t.is_stop, t.lemma_, clase) for doc, clase in zip(docs, df["class"]) for t in doc],
    columns=["pos", "stop", "lema", "clase"])

# Distribución de categorías gramaticales.
distribucion = tokens["pos"].value_counts()
ax = distribucion.plot.bar(figsize=(8, 3.2), color="#4C72B0")
ax.set_title("Categorías gramaticales detectadas en el corpus")
ax.set_ylabel("tokens")
plt.tight_layout()
plt.show()
print(f"Sin categoría (X): {distribucion.get('X', 0) / len(tokens):.1%} · stop-words: {tokens['stop'].mean():.1%}")
print("Tokens más frecuentes sin categoría:", tokens.loc[tokens["pos"] == "X", "lema"].value_counts().head(12).to_dict())

# Los 8 lemas de contenido (sustantivos, nombres propios, verbos y adjetivos que no son stop-words)
# más frecuentes de cada clase: si el modelo funciona, deben tener sentido temático.
contenido = tokens[tokens["pos"].isin({"NOUN", "PROPN", "VERB", "ADJ"}) & ~tokens["stop"]]
display(pd.DataFrame({clase: g["lema"].value_counts().head(8).index.tolist()
                      for clase, g in contenido.groupby("clase")}))

## Fase 3 · Clasificación

### Representación tabular y entrenamiento

**Representación.** Un modelo de aprendizaje automático necesita números, no texto. Cada párrafo pasa por el modelo de la Fase 2 y se queda con los lemas de sus palabras de contenido (sustantivos, nombres propios, verbos y adjetivos que no son stop-words). Con esos lemas se construye una matriz **TF-IDF**: una fila por párrafo y una columna por término (unigramas y bigramas), descartando los términos que aparecen en menos de 2 párrafos. El lema evita que `soldato` y `soldatoj` cuenten como palabras distintas, y TF-IDF resta peso a las palabras que aparecen en todas las clases.

**Con y sin números.** Los años (1799–1815, 1936–1939, 1939–1945) casi delatan la clase, y eso no demuestra que el modelo entienda el tema. Se entrena con las dos variantes y se comparan.

**Metodología.**
1. Partición 80 % / 20 % estratificada (mantiene la proporción de clases) y con semilla fija. El 20 % (test) no se toca hasta la evaluación final.
2. Tres algoritmos baratos (Naive Bayes, regresión logística y SVM lineal) se comparan con validación cruzada de 5 particiones usando **solo el 80 %**. El vectorizador va dentro de un `Pipeline`, así que se ajusta en cada partición y no filtra información del test.
3. El modelo final se elige por F1 macro en validación cruzada entre los que dan probabilidades, porque la clasificación de páginas web las necesita.
4. Evaluación en test: accuracy, F1 macro y matriz de confusión.

In [ ]:
# Categorías gramaticales que aportan significado temático.
CONTENIDO = {"NOUN", "PROPN", "VERB", "ADJ"}


def a_texto(doc, con_numeros=True):
    """Convierte un Doc en un texto de lemas, que es lo que recibe el vectorizador.

    Parámetros
    ----------
    doc : Doc
        Resultado de nlp(texto).
    con_numeros : bool
        True: se conservan también los números (años, fechas). False: se quitan.

    Devuelve
    --------
    str con los lemas separados por espacios, por ejemplo "flago ruĝa nigra vertikala strio".

    Cómo funciona
    -------------
    Se queda con los tokens cuya categoría está en CONTENIDO (más "NUM" si con_numeros),
    que no son stop-words y cuyo lema tiene más de una letra. " ".join(lista) une los
    lemas con espacios.
    """
    categorias = (CONTENIDO | {"NUM"}) if con_numeros else CONTENIDO
    lemas = [t.lemma_ for t in doc
             if t.pos_ in categorias and not t.is_stop and len(t.lemma_) > 1]
    return " ".join(lemas)


# Una fila por párrafo: el texto original, sus lemas con y sin números, y su clase.
datos = pd.DataFrame({
    "parrafo": df["text"],
    "lemas_con": [a_texto(doc, con_numeros=True) for doc in docs],
    "lemas_sin": [a_texto(doc, con_numeros=False) for doc in docs],
    "clase": df["class"],
})

# Partición 80 % entrenamiento / 20 % test. stratify=... mantiene la proporción de clases en ambos.
# random_state=SEMILLA hace que la partición sea siempre la misma.
train, test = train_test_split(datos, test_size=0.2, stratify=datos["clase"], random_state=SEMILLA)
print(f"Entrenamiento: {len(train)} párrafos · test: {len(test)} párrafos")
print("Párrafo original :", datos.loc[0, "parrafo"][:150])
print("Lemas (con nº)   :", datos.loc[0, "lemas_con"][:150])
print("Lemas (sin nº)   :", datos.loc[0, "lemas_sin"][:150])

In [ ]:
def crear_modelo(nombre):
    """Crea un modelo NUEVO (sin entrenar): vectorizador TF-IDF + clasificador.

    Parámetros
    ----------
    nombre : str
        "Naive Bayes", "Regresión logística" o "SVM lineal".

    Devuelve
    --------
    sklearn Pipeline: encadena TfidfVectorizer y el clasificador. fit(textos, clases) ajusta
    primero el vectorizador y luego el clasificador; predict(textos) hace las dos cosas a la inversa.

    Parámetros del vectorizador
    ---------------------------
    ngram_range=(1, 2)  términos de una palabra y de dos seguidas ("primo rivera")
    min_df=2            se ignoran los términos que aparecen en menos de 2 párrafos
    sublinear_tf=True   el peso crece con el logaritmo de las repeticiones, no linealmente

    Los tres algoritmos entrenan en segundos y sin GPU. Meter el vectorizador dentro del
    Pipeline hace que, en la validación cruzada, se ajuste solo con los datos de entrenamiento
    de cada partición y no se filtre información del test.
    """
    clasificadores = {
        "Naive Bayes": MultinomialNB(alpha=0.1),   # alpha: suavizado para palabras no vistas
        "Regresión logística": LogisticRegression(C=10, max_iter=1000, random_state=SEMILLA),
        "SVM lineal": LinearSVC(C=1, random_state=SEMILLA),
    }
    vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    return make_pipeline(vectorizador, clasificadores[nombre])


NOMBRES_MODELOS = ["Naive Bayes", "Regresión logística", "SVM lineal"]
CON_PROBABILIDADES = {"Naive Bayes", "Regresión logística"}      # la SVM lineal no da probabilidades
VARIANTES = {"con números": "lemas_con", "sin números": "lemas_sin"}    # variante -> columna de `datos`

# Validación cruzada de 5 particiones SOLO con el conjunto de entrenamiento: el entrenamiento se
# divide en 5 partes; cada parte hace de validación una vez mientras las otras 4 entrenan.
# La métrica es el F1 macro (media del F1 de cada clase), que trata igual a todas las clases.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

filas = []
for nombre in NOMBRES_MODELOS:
    for variante, columna in VARIANTES.items():
        puntuaciones = cross_val_score(crear_modelo(nombre), train[columna], train["clase"],
                                       cv=cv, scoring="f1_macro")    # 5 puntuaciones, una por partición
        filas.append({"modelo": nombre, "variante": variante,
                      "F1 macro (CV)": puntuaciones.mean(), "desviación": puntuaciones.std()})

cv_resultados = pd.DataFrame(filas)
display(cv_resultados.round(3))

### Modelo final y evaluación en test

Se elige el modelo con mejor F1 macro en validación cruzada (nunca por el test), se entrena con todo el conjunto de entrenamiento y se evalúa una sola vez con el test. Además se muestran las palabras que más pesan en cada clase y algunos errores, para entender qué ha aprendido el modelo.

In [ ]:
# Mejor combinación (modelo, variante) por F1 macro en validación cruzada, solo entre los modelos que dan probabilidades.
elegido = (cv_resultados[cv_resultados["modelo"].isin(CON_PROBABILIDADES)]
           .sort_values("F1 macro (CV)", ascending=False).iloc[0])
NOMBRE_MODELO = elegido["modelo"]
CON_NUMEROS = elegido["variante"] == "con números"
COLUMNA = VARIANTES[elegido["variante"]]
print(f"Modelo elegido: {NOMBRE_MODELO}, {elegido['variante']} (F1 macro en CV = {elegido['F1 macro (CV)']:.3f})")

# Entrenamiento con todo el conjunto de entrenamiento y predicción sobre el test.
modelo_final = crear_modelo(NOMBRE_MODELO).fit(train[COLUMNA], train["clase"])
prediccion = modelo_final.predict(test[COLUMNA])

print(f"\nAccuracy = {accuracy_score(test['clase'], prediccion):.3f}"
      f" · F1 macro = {f1_score(test['clase'], prediccion, average='macro'):.3f}\n")
print(classification_report(test["clase"], prediccion, digits=3))

# Matriz de confusión: filas = clase real, columnas = clase predicha. La diagonal son los aciertos.
fig, ax = plt.subplots(figsize=(5.6, 4.4))
ConfusionMatrixDisplay.from_predictions(test["clase"], prediccion, ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión (test)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


def palabras_clave(modelo, n=10):
    """Las n palabras que más distinguen a cada clase en un modelo entrenado.

    Parámetros
    ----------
    modelo : Pipeline entrenado (TF-IDF + Naive Bayes o regresión logística).
    n : int
        Cuántas palabras se muestran por clase.

    Devuelve
    --------
    DataFrame con una columna por clase.

    Cómo funciona
    -------------
    Cada clasificador guarda un peso por término y clase (coef_ en la regresión logística,
    feature_log_prob_ en Naive Bayes). Se resta la media de las clases para quedarse con lo que
    hace a un término más propio de una clase que de las demás, y se ordenan de mayor a menor.
    """
    vocabulario = np.array(modelo[0].get_feature_names_out())    # modelo[0] = el vectorizador
    clasificador = modelo[-1]                                    # modelo[-1] = el clasificador
    pesos = clasificador.coef_ if hasattr(clasificador, "coef_") else clasificador.feature_log_prob_
    pesos = pesos - pesos.mean(axis=0)
    return pd.DataFrame({clase: vocabulario[np.argsort(pesos[i])[::-1][:n]]   # argsort ordena de menor a mayor; [::-1] lo invierte
                         for i, clase in enumerate(clasificador.classes_)})


display(palabras_clave(modelo_final))

# Errores: suelen ser párrafos genéricos que heredaron la etiqueta de su artículo (ruido de etiquetas).
resultados = test.assign(prediccion=prediccion)
errores = resultados[resultados["clase"] != resultados["prediccion"]]
print(f"\n{len(errores)} errores de {len(test)}. Ejemplos (clase real -> predicha):")
for _, fila in errores.head(4).iterrows():
    print(f" - {fila['clase']} -> {fila['prediccion']}: {fila['parrafo'][:140]}...")

### Clasificación de páginas web reales

Para probar el modelo fuera del dataset se descargan páginas de Vikipedio que **no** están en él (se comprobó que sus títulos no aparecen en `data/raw/textos.json`), se extrae su texto con *web scraping* y se clasifica cada párrafo. La probabilidad de la página es la media de las probabilidades de sus párrafos. La clase esperada solo sirve para valorar el resultado: el modelo no la ve.

Si una página no se puede descargar se omite con un aviso, y si fallan todas se usan unos textos de respaldo. Así el cuaderno nunca se rompe por una web caída.

In [ ]:
URL_WIKI = "https://eo.wikipedia.org/wiki/"      # las páginas se piden como URL_WIKI + título con "_" en vez de espacios

# Páginas de prueba: título -> clase esperada.
PAGINAS_WEB = {
    "Batalo de Stalingrado": "Segunda Guerra Mundial",
    "Atako kontraŭ Pearl Harbor": "Segunda Guerra Mundial",
    "Francisko Franko": "Guerra Civil Española",
    "Batalo de Waterloo": "Guerras Napoleónicas",
    "Batalo de Trafalgar": "Guerras Napoleónicas",
    "Napoleono Bonaparte": "Guerras Napoleónicas",
}

# Si no se pudiera descargar ninguna página, el cuaderno sigue funcionando con estos textos.
# Formato: nombre -> (clase esperada, lista de párrafos).
TEXTOS_RESPALDO = {
    "Texto de respaldo (2ª G. M.)": ("Segunda Guerra Mundial", [
        "La Dua Mondmilito komenciĝis en 1939, kiam Germanio invadis Pollandon, kaj finiĝis en 1945 per la kapitulaco de Germanio kaj Japanio.",
        "Dum la milito la aliancanoj batalis kontraŭ la Akso en Eŭropo, Afriko kaj Azio, kaj milionoj da civiluloj mortis en bombadoj kaj koncentrejoj."]),
    "Texto de respaldo (G. C. E.)": ("Guerra Civil Española", [
        "La Hispana Enlanda Milito komenciĝis en 1936 post la ribelo de generalo Franko kontraŭ la respublika registaro kaj daŭris ĝis 1939.",
        "La Internaciaj Brigadoj subtenis la respublikanojn ĉe la batalo de Madrido, dum Germanio kaj Italio helpis la ribelantojn per aviadiloj kaj soldatoj."]),
    "Texto de respaldo (G. Napoleónicas)": ("Guerras Napoleónicas", [
        "Napoleono Bonaparte venkis en la batalo de Austerlitz en 1805, kaj lia franca armeo invadis Hispanion en 1808, kio ekigis longan gerilan militon.",
        "Post la malvenko en Rusio kaj la batalo de Waterloo en 1815, la imperiestro estis ekzilita al la insulo Sankta Heleno."]),
}


def extraer_parrafos(html):
    """Extrae los párrafos de una página de Vikipedio, ya limpios.

    Parámetros
    ----------
    html : str
        Código HTML de la página.

    Devuelve
    --------
    list[str] con los párrafos del contenido principal que pasan parrafo_limpio().

    Cómo funciona (BeautifulSoup)
    -----------------------------
    1. BeautifulSoup(html, "html.parser") convierte el HTML en un árbol de etiquetas.
    2. soup.find("div", {"id": "mw-content-text"}) busca el primer <div> con ese id, que es
       donde Vikipedio pone el texto del artículo (sin menús ni barras laterales). Si no
       existe se usa toda la página.
    3. decompose() borra una etiqueta del árbol. Se borran las referencias (<sup>), los estilos
       y scripts, las tablas y los enlaces "editar" para que su texto no se mezcle con el del artículo.
    4. find_all("p") devuelve todos los párrafos <p>; .text da su texto sin etiquetas.
    """
    sopa = BeautifulSoup(html, "html.parser")
    zona = sopa.find("div", {"id": "mw-content-text"}) or sopa
    for etiqueta in zona.find_all(["sup", "style", "script", "table"]) + zona.find_all(class_="mw-editsection"):
        etiqueta.decompose()
    textos = [p.text for p in zona.find_all("p")]
    return [p for p in map(parrafo_limpio, textos) if p is not None]


def clasificar_pagina(parrafos):
    """Probabilidad de cada clase para una página, a partir de sus párrafos.

    Parámetros
    ----------
    parrafos : list[str]
        Párrafos de la página.

    Devuelve
    --------
    pandas.Series {clase: probabilidad media}.

    Cómo funciona
    -------------
    Cada párrafo recibe el MISMO preprocesado que en el entrenamiento: nlp(...) -> a_texto(...).
    predict_proba da, por cada párrafo, una probabilidad por clase (suman 1); se hace la media
    por columnas. modelo_final.classes_ da el orden de las clases.
    """
    lemas = [a_texto(nlp(p), CON_NUMEROS) for p in parrafos]
    probabilidades = modelo_final.predict_proba(lemas)           # matriz: una fila por párrafo, una columna por clase
    return pd.Series(probabilidades.mean(axis=0), index=modelo_final.classes_)


# 1) Descarga y extracción. Cada fallo se salta sin detener el cuaderno.
fuentes = {}                                       # título -> (clase esperada, párrafos)
for titulo, esperada in PAGINAS_WEB.items():
    respuesta = pedir(URL_WIKI + titulo.replace(" ", "_"))
    parrafos = extraer_parrafos(respuesta.text) if respuesta is not None else []
    if parrafos:
        fuentes[titulo] = (esperada, parrafos)
    else:
        print(f"Aviso: no se pudo obtener «{titulo}»; se omite.")
if not fuentes:
    print("Aviso: ninguna página disponible; se usan los textos de respaldo.")
    fuentes = TEXTOS_RESPALDO

# 2) Clasificación: la clase predicha es la de mayor probabilidad media.
filas, perfiles = [], {}
for titulo, (esperada, parrafos) in fuentes.items():
    perfil = clasificar_pagina(parrafos)
    predicha = perfil.idxmax()                     # idxmax = nombre de la clase con mayor probabilidad
    perfiles[titulo] = perfil
    filas.append({"página": titulo, "párrafos": len(parrafos), "clase esperada": esperada,
                  "clase predicha": predicha, "probabilidad": round(perfil.max(), 3),
                  "acierto": "✔" if predicha == esperada else "✘"})
display(pd.DataFrame(filas))

# 3) Probabilidad de cada clase por página (barras apiladas: cada barra suma 1).
ax = pd.DataFrame(perfiles).T.plot.barh(stacked=True, figsize=(9, 3.6), width=0.7,
                                        color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.set_xlabel("probabilidad media de los párrafos")
ax.set_title("Clasificación de páginas web en esperanto")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
plt.tight_layout()
plt.show()

### Resultados y conclusiones

**Fase 2 · modelo de NLP.** Las pruebas con `assert` fijan el análisis esperado de casos difíciles (`milito`, participios, nombres propios, ordinales sin guion). Sobre los 1.803 párrafos del dataset, solo el 1,6 % de los tokens se queda sin categoría (`X`) y son casi todo palabras extranjeras (`von`, `und`, `del`, `of`...) o abreviaturas. Los lemas más frecuentes de cada clase tienen sentido temático (`hispana`, `respublika`, `napoleono`, `germana`...).

**Fase 3 · entrenamiento y evaluación.**
- **Validación cruzada (solo entrenamiento).** Naive Bayes obtiene el mejor F1 macro, 0,860 con números y 0,848 sin ellos; la regresión logística y la SVM lineal quedan en torno a 0,83–0,84. Se elige Naive Bayes con números porque da probabilidades.
- **Test (361 párrafos).** Accuracy 0,861 y F1 macro 0,861, igual que en validación cruzada: no hay señales de sobreajuste ni de fuga de información.
- **Matriz de confusión.** Los fallos más frecuentes llevan hacia la Guerra Civil Española (13 de la Segunda Guerra Mundial y 10 de las Guerras Napoleónicas), la clase con menos artículos y menos variedad de temas.
- **Números.** Quitarlos cuesta aproximadamente un punto de F1 en validación cruzada. Los años ayudan (`1812`, `1808` y `1936` están entre las palabras más discriminativas), pero no son lo que sostiene el modelo.
- **Palabras que distinguen cada clase.** `frankisma`, `respublika` y `primo rivera` para la Guerra Civil Española; `napoleono` y `bonaparte` para las Guerras Napoleónicas; `bombardadi`, `japano` y `usonano` para la Segunda Guerra Mundial.
- **Errores y ruido de etiquetas.** Hay 50 errores de 361 (14 %). Los ejemplos mostrados son párrafos que no hablan de la guerra (bibliografía, genealogía, enlaces de archivo) y heredan la etiqueta del artículo, lo que confirma el ruido del que advertía el enunciado.

**Páginas web.** Las 6 páginas de Vikipedio que no están en el dataset se clasifican en la clase esperada, con probabilidades entre 0,60 y 0,97. Las más bajas (Waterloo y Francisko Franko) indican un margen menor y no se ha analizado su causa.

**Limitaciones.**
- Las páginas de prueba proceden de la misma fuente que el entrenamiento (Vikipedio). No se ha medido el comportamiento con otros estilos de web.
- La clase esperada de cada página la fijamos nosotros según su tema; la de Francisko Franko es discutible porque su biografía cubre también la dictadura.
- Un par de páginas tienen pocos párrafos de 10 palabras o más (Stalingrado solo 3), así que su probabilidad es menos fiable.